# Unidad 39. Seaborn, datos y gramática visual

**Bloque:** Seaborn (UT3) · **Tiempo orientativo:** 5–7 horas

> En esta unidad usarás Seaborn para representar un `DataFrame` local. Seaborn no sustituye a Pandas ni a Matplotlib: se apoya en ambas bibliotecas para expresar visualizaciones estadísticas de forma concisa.

## Resultados de aprendizaje

Al terminar podrás:

- cargar un CSV local y comprobar qué representa cada columna;
- distinguir una función de Seaborn de nivel de ejes de una de nivel de figura;
- relacionar las columnas de un `DataFrame` con los mapeos visuales `x`, `y`, `hue`, `style` y `size`;
- crear una visualización legible y describir con prudencia el patrón que muestra.

**Prerequisitos:** unidades 18–23 (Pandas) y 31–35 (Matplotlib). Necesitas tener instalados los paquetes indicados en `requirements.txt`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

print(f"Pandas: {pd.__version__}")
print(f"Seaborn: {sns.__version__}")
print(f"Matplotlib: {plt.matplotlib.__version__}")

## 1. Qué aporta Seaborn

Pandas sirve para cargar, inspeccionar y transformar tablas. Matplotlib permite controlar con detalle figuras y ejes. **Seaborn** ofrece funciones que entienden directamente columnas de un `DataFrame` y convierten sus relaciones en marcas visuales: puntos, líneas, barras o distribuciones.

Un gráfico no analiza los datos por sí solo. Antes de elegir una función, formula una pregunta concreta, identifica las columnas necesarias y decide qué comparación sería honesta. En las unidades siguientes veremos tipos concretos de gráficos; aquí aprenderás el lenguaje común con el que Seaborn recibe los datos.

## 2. Datos locales, rutas portables y primera inspección

El curso no dependerá de los conjuntos de datos descargados por Seaborn. El archivo `39_registros_entrenamiento.csv` es local y sintético: contiene dos ejecuciones simuladas de entrenamiento. La función siguiente funciona si abres el cuaderno desde la carpeta del curso o desde `unidades/`.

No interpretes `modelo` como una demostración de que una configuración sea mejor en general: solo identifica dos series de este ejemplo. Antes de visualizar, comprueba siempre las columnas, los tipos y los valores ausentes.

In [ ]:
def ruta_datos(nombre):
    """Busca un dato del bloque Seaborn desde rutas habituales del curso."""
    candidatas = [
        Path("datos") / "seaborn" / nombre,
        Path("..") / "datos" / "seaborn" / nombre,
    ]
    for candidata in candidatas:
        if candidata.exists():
            return candidata
    raise FileNotFoundError(
        f"No se encontró {nombre}. Abre el cuaderno desde la carpeta del curso."
    )

registros = pd.read_csv(ruta_datos("39_registros_entrenamiento.csv"))
registros.info()
registros.head()

### Comprueba que lo has entendido

Sin ejecutar de nuevo la celda anterior, responde en una celda Markdown o en tu cuaderno: ¿qué columnas son numéricas y cuál podría servir para separar observaciones en grupos? Después contrasta tu respuesta con la salida de `info()`.

Una fila representa una época de una ejecución. Como `epoca` tiene orden, una línea puede ser útil para seguir su evolución. En cambio, `modelo` no tiene un orden numérico natural: se usa para identificar grupos.

## 3. La gramática visual: datos que se convierten en señales

En muchas funciones de Seaborn se pasan nombres de columnas, no series sueltas. Cada parámetro asigna una propiedad visual:

- `x` e `y`: posición horizontal y vertical.
- `hue`: color para distinguir grupos. Debe añadirse solo si responde a una comparación útil.
- `style`: forma de línea o marcador; puede reforzar una diferencia sin depender solo del color.
- `size`: tamaño de una marca. Es fácil exagerar diferencias con él, así que úsalo con cuidado.

El argumento `data=registros` indica de qué tabla salen esos nombres. Este patrón evita repetir `registros["columna"]` y permite que Seaborn trate los grupos de forma coherente.

In [ ]:
sns.set_theme(style="whitegrid", context="notebook")

fig, ax = plt.subplots(figsize=(7, 4))
sns.lineplot(
    data=registros,
    x="epoca",
    y="perdida_validacion",
    hue="modelo",
    style="modelo",
    marker="o",
    ax=ax,
)
ax.set(
    title="Pérdida de validación por época",
    xlabel="Época",
    ylabel="Pérdida de validación",
)
ax.legend(title="Configuración")
plt.show()

La figura muestra una pérdida menor en la serie `ajustado` para estas observaciones. No permite concluir que esa configuración sea siempre superior: faltan más repeticiones, información sobre los datos y una definición completa del experimento. La visualización describe lo observado; la explicación causal requiere evidencias adicionales.

**Error frecuente:** asignar `hue` a una columna numérica sin pensar en la leyenda. Seaborn puede crear muchos colores y una leyenda ilegible. Para comparar dos grupos, una columna categórica como `modelo` es más adecuada.

## 4. Dos familias de funciones: nivel de ejes y nivel de figura

`sns.lineplot` es una función de **nivel de ejes**: dibuja dentro del objeto `Axes` que le pases con `ax=`. Por eso encaja bien en una figura de Matplotlib con varios paneles.

`sns.relplot` es una función de **nivel de figura**: crea y administra una figura completa. Resulta útil cuando más adelante quieras dividir los datos en paneles. En ambas familias, la pregunta y los mapeos visuales siguen siendo lo importante.

In [ ]:
grafico = sns.relplot(
    data=registros,
    kind="scatter",
    x="perdida_validacion",
    y="exactitud_validacion",
    hue="modelo",
    style="modelo",
    height=4,
    aspect=1.35,
)
grafico.set_axis_labels("Pérdida de validación", "Exactitud de validación")
grafico.figure.suptitle("Dos medidas de las ejecuciones simuladas", y=1.03)
plt.show()

Observa que la variable `grafico` no es un `Axes`: es un objeto de nivel de figura que contiene una figura y sus ejes. En el ejemplo anterior usar `grafico.set_axis_labels(...)` es más apropiado que llamar a `ax.set_xlabel(...)`, porque no existe una variable `ax` creada por `relplot`.

No memorices esta diferencia a ciegas. Pregúntate primero si necesitas integrar el gráfico en una figura de Matplotlib ya construida (nivel de ejes) o si Seaborn debe crear la figura completa (nivel de figura).

## Práctica guiada · una decisión cada vez

Parte del código de la primera gráfica. Haz estas modificaciones en orden:

1. Sustituye `perdida_validacion` por `exactitud_validacion` en `y`.
2. Mantén `hue` y `style` para que las dos series sigan siendo distinguibles sin depender únicamente del color.
3. Actualiza el título y la etiqueta vertical para que indiquen correctamente la magnitud.
4. Explica en un comentario qué serie tiene mayor exactitud en la última época y qué información faltaría para hacer una afirmación general.

La salida debe mostrar dos líneas, seis puntos por línea y una leyenda con dos configuraciones.

In [ ]:
# Completa la práctica guiada aquí.
# Reutiliza registros, fig y ax; no cargues datos de Internet.


## Ejercicios

Intenta cada ejercicio antes de consultar el cuaderno de soluciones. Además de ejecutar el código, revisa que el título, las etiquetas y la leyenda respondan a la pregunta planteada.

### Ejercicio 39.01 · Comprobación de los datos

Escribe código que compruebe estas condiciones y lance un error claro si alguna falla: hay 12 filas, no hay valores ausentes y las columnas `epoca`, `modelo`, `perdida_validacion` y `exactitud_validacion` existen. Muestra después las filas ordenadas por `modelo` y `epoca`.

In [ ]:
# Escribe aquí tu solución del ejercicio 39.01.


### Ejercicio 39.02 · Mapeos con intención

Crea un diagrama de dispersión de `tiempo_s` frente a `exactitud_validacion`. Usa `hue="modelo"` y `style="modelo"`; añade título, etiquetas y leyenda. En un comentario, indica por qué no usas `size="epoca"` en esta figura.

In [ ]:
# Escribe aquí tu solución del ejercicio 39.02.


### Ejercicio 39.03 · Una función de nivel de ejes

Define `grafico_por_modelo(datos, columna, etiqueta_y)`. Debe crear `fig, ax` con `plt.subplots`, dibujar una línea de `columna` por época con color y estilo por modelo, etiquetar los ejes y devolver `fig, ax`. Compruébala con `columna="tiempo_s"` y verifica que `len(fig.axes) == 1`.

In [ ]:
# Escribe aquí tu solución del ejercicio 39.03.


### Ejercicio 39.04 · Figura completa con Seaborn

Construye un `relplot` de tipo `line` para mostrar la pérdida por época y modelo. Pon el resultado en una variable llamada `grafico_perdida`. Ajusta etiquetas y título mediante métodos del objeto de nivel de figura. Explica en un comentario una situación en la que preferirías `lineplot(..., ax=ax)`.

In [ ]:
# Escribe aquí tu solución del ejercicio 39.04.


### Ejercicio 39.05 · Miniproyecto: evolución de una medida de validación

Una compañera necesita responder: **«¿Cómo evolucionó la exactitud de validación de cada configuración en estas ejecuciones simuladas?»**

Crea una figura de nivel de ejes que responda a la pregunta. Debe incluir dos líneas distinguibles por color y estilo, marcadores, título, etiquetas, leyenda y cuadrícula horizontal. Bajo la celda, redacta entre dos y cuatro frases que separen el patrón observado de lo que no puede concluirse con estos datos.

**Ampliación voluntaria:** exporta la figura a `unidades/salidas/39_exactitud_validacion.png` creando la carpeta si fuera necesario. No entregues la imagen generada al repositorio.

In [ ]:
# Escribe aquí tu solución del ejercicio 39.05.


## Autoevaluación y cierre

Antes de continuar, comprueba que puedes responder sin ejecutar código:

1. ¿Qué argumento indica a Seaborn el `DataFrame` del que salen los nombres de columnas?
2. ¿Para qué usarías `hue` y por qué no debe ser la única señal que distinga grupos?
3. ¿Qué diferencia práctica hay entre `sns.lineplot(..., ax=ax)` y `sns.relplot(...)`?
4. Una gráfica muestra menor pérdida en una serie: ¿qué afirmación prudente puedes hacer y cuál no?

En la siguiente unidad estudiarás cómo representar distribuciones de una variable. Conserva la costumbre de inspeccionar los datos y formular una pregunta antes de elegir un gráfico.